In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys
import pandas as pd
import datetime as dt

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

from utils.validator import Validator, error_masks, warning_masks

In [3]:
merge_df = pd.read_pickle("251021_merge_df.pkl")
print(len(merge_df))

5870


In [4]:
[c for c in merge_df.columns if "year" in c]

['plan_year_id',
 'plan_year.id',
 'plan_year.organization_id',
 'plan_year.name',
 'plan_year.valid_from',
 'plan_year.valid_to',
 'plan_year.organization_path',
 'plan_year.prior_plan_year_id',
 'plan_year.prior_plan_year.id',
 'plan_year.prior_plan_year.name',
 'plan_year.prior_plan_year.valid_from',
 'plan_year.prior_plan_year.valid_to',
 'prior_plan.plan_year.id',
 'prior_plan.plan_year.name',
 'year_oe_approved',
 'date_renewal_year_begins',
 'date_renewal_year_ends']

In [5]:
jan_plans = merge_df[
    (merge_df['plan_year.valid_to'] > dt.datetime(2025, 12, 30)) &
    (merge_df['plan_year.valid_to'] <= dt.datetime(2026, 1, 1))
]

In [6]:
print(f'{len(jan_plans)} January Plans')

valid_df = Validator(jan_plans, error_masks, warning_masks)
valid_df = valid_df.validate()

error_counts = valid_df['error'].explode().value_counts()
display(error_counts)

warning_counts = valid_df['warning'].explode().value_counts()
display(warning_counts)

non_error = valid_df[valid_df['error'].apply(lambda x: len(x) == 0)]


2438 January Plans


error
non-active organization     58
auto-renew not turned on    39
short plan year             28
non-active plan             12
`template` in plan name      6
Name: count, dtype: int64

warning
no `cu_plan_id` found    547
no `client_id` found       3
Name: count, dtype: int64

In [18]:
[c for c in valid_df if "status" in c]

['elv_plan_status',
 'prior_plan.plan_status',
 'organization_status_type',
 'organization_status_valid_from',
 'cu_client_status',
 'status.id_x',
 'status.color_x',
 'status.type_x',
 'status.orderindex_x',
 'cu_plan_status',
 'status.id_y',
 'status.color_y',
 'status.type_y',
 'status.orderindex_y']

In [ ]:
no_auto = valid_df[
    (valid_df['error'].apply(lambda x: "auto-renew not turned on" in x)) &
    (valid_df['cu_client_status'].apply(lambda x: pd.notna(x) and"active" in x))
]['client_id'].unique()

In [26]:
list(no_auto)

['868cpf8fc']

In [ ]:
non_error['plan_year.valid_from'].value_counts()

In [ ]:
valid_df[valid_df['error'].apply(lambda x: len(x) > 0)]